In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk
import pandas as pd

In [3]:
df = pd.read_csv('/content/dataset.csv')

In [4]:
df.head(3)

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,0
1,Army chiefs in regiments decision\n \n Militar...,0
2,Howard denies split over ID cards\n \n Michael...,0


In [5]:
df.drop(columns=['Label'],inplace=True)

In [6]:
df.head(2)

,Text
0,Budget to set scene for election\n \n Gordon B...
1,Army chiefs in regiments decision\n \n Militar...


In [7]:
document = ''
def extract_text(row):
  global document
  document = document + row['Text'] + ' /n '


In [8]:
df.apply(extract_text,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
2220,None
2221,None
2222,None
2223,None


In [9]:
len(document)

5071584

In [10]:
 # Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
tokens = word_tokenize(document.lower())

In [12]:
voc = {'<UNK>': 0}
count = 1
for token in tokens:
  if token not in voc:
    voc[token] = count
    count +=1

len(voc)

34615

In [13]:
input_sentences = [x for x in document.split('\n') if x.strip()]

len(input_sentences)

12861

In [14]:
def text_to_index(text):
  arr = []
  for token in text:
    if token in voc:
      arr.append(voc[token])
    else:
      arr.append(voc['<UNK>'])
  return arr

In [15]:
input_indexes = []
for text in input_sentences:
  input_indexes.append(text_to_index(word_tokenize(text.lower())))

len(input_indexes)

12861

In [16]:
input_indexes[10]

[293, 294, 295, 23, 296, 297]

In [17]:
training_seq = []

for indexes in input_indexes:
  for i in range(len(indexes)):
    training_seq.append(indexes[:i+1])

len(training_seq)

976554

In [18]:
print(training_seq[0]," ->", training_seq[1])

[1]  -> [1, 2]


In [19]:
max_length = 0
for seq in training_seq:
  if len(seq) > max_length:
    max_length = len(seq)

max_length

515

In [20]:
for i in range(0,len(training_seq)):
  training_seq[i] = [0]*(max_length-len(training_seq[i])) + training_seq[i]

In [21]:
training_seq = torch.tensor(training_seq,dtype = torch.long)

In [22]:
X = training_seq[:,:-1]
y = training_seq[:,-1]

In [23]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [24]:
dataset = CustomDataset(X,y)

In [25]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


In [26]:
class LSTM(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,100)
    self.lstm = nn.LSTM(100,150,batch_first=True)
    self.fc = nn.Linear(150,vocab_size)


  def forward(self, x):
    embed = self.embedding(x)
    output, (hidden, cell) = self.lstm(embed)
    final_output = self.fc(hidden.squeeze(0))
    return final_output

In [27]:
model = LSTM(len(voc))

In [28]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

LSTM(
  (embedding): Embedding(34615, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=34615, bias=True)
)

In [29]:
epochs = 25
learning_rate = 0.001

In [30]:

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:


for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

In [ ]:


def prediction(model, vocab, text):

  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_index(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (max_length - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]

